# Неделя 4. Экспериментальное исследование и анализ ошибок

Продолжение недели 3: тот же корпус (8 кодексов РК), та же задача (по тексту статьи определить кодекс), то же тестовое разбиение (seed 42), те же модель и настройки обучения. В центре сетки экспериментов стоят гиперпараметры недели 3.

**Что делает ноутбук**

1. Меняет один гиперпараметр, скорость обучения (в сетке три значения: треть, 1x и 5/3 от значения недели 3). Для модели `rubert-base-cased` это 1e-5, 3e-5 и 5e-5, как в задании.
2. Для лучшей скорости обучения (выбор делается по валидационной выборке, а не по тесту) обучает модель на 50%, 75% и 100% обучающих данных.
3. Считает полный набор метрик: Accuracy, Precision, Recall, F1 (macro-усреднение по классам).
4. Разбирает ошибки лучшей модели: таблица не менее чем из 10 ошибок, группировка причин, предложения по исправлению.
5. Сохраняет таблицы, графики и черновик аналитического отчёта в `report/`.

**Про «базовую модель с 3-й недели».** Каждый эксперимент стартует с предобученных весов, а не с модели, уже дообученной на неделе 3: иначе скорости обучения сравнивались бы нечестно (модель уже видела эти данные). Конфигурация недели 3 при этом входит в сетку как центральная точка. Если рядом лежит `report/week3_hyperparameters.json`, ноутбук берёт из него модель и параметры.

**Против утечки данных.** Тестовая выборка та же, что на неделе 3. Из обучающей части выделено 10% на валидацию: по ней выбирается лучшая скорость обучения. Тест используется только для итоговых цифр и анализа ошибок.

**Время.** Всего 5 дообучений: 3 для сетки скоростей и 2 для уменьшенных выборок. Результаты каждого сохраняются в `report/week4_runs.json`, поэтому при обрыве ноутбук можно запустить заново, и завершённые эксперименты будут взяты из кэша.

In [1]:
# Локально torch нужно поставить заранее: pip install torch
%pip install -q transformers scikit-learn pandas matplotlib

Note: you may need to restart the kernel to use updated packages.


## 1. Настройки и данные

In [2]:
import gc
import json
import logging
import random
import re
import time
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score)
from sklearn.model_selection import train_test_split
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          get_linear_schedule_with_warmup)

logging.getLogger("transformers.tokenization_utils_base").setLevel(logging.ERROR)

SEED = 42
TEST_SIZE = 0.2      # как на неделе 3
VAL_SIZE = 0.1       # доля обучающей части, отданная на валидацию
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Настройки берём из недели 3, если она была запущена
EPOCHS, BATCH_SIZE, MAX_LEN = 3, 16, 256
if DEVICE == "cuda":
    MODEL_NAME, BASE_LR = "DeepPavlov/rubert-base-cased", 3e-5
else:
    MODEL_NAME, BASE_LR = "cointegrated/rubert-tiny2", 1e-4

W3 = next((p for p in [Path("report/week3_hyperparameters.json"),
                       Path("week3_hyperparameters.json")] if p.exists()), None)
if W3 is not None:
    cfg = json.loads(W3.read_text(encoding="utf-8"))
    MODEL_NAME, BASE_LR = cfg["model"], cfg["learning_rate"]
    EPOCHS, BATCH_SIZE, MAX_LEN = cfg["epochs"], cfg["batch_size"], cfg["max_len"]
    if DEVICE == "cpu" and "tiny" not in MODEL_NAME:
        print("Предупреждение: в неделе 3 была большая модель, а видеокарты нет. "
              "Переключаюсь на rubert-tiny2, иначе обучение займёт часы.")
        MODEL_NAME, BASE_LR = "cointegrated/rubert-tiny2", 1e-4


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)
print("Устройство:", DEVICE, "| модель:", MODEL_NAME, "| lr недели 3:", BASE_LR)
print("Эпох:", EPOCHS, "| batch:", BATCH_SIZE, "| max_len:", MAX_LEN)

c:\Users\Acer\Desktop\nlp-legal-project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Устройство: cpu | модель: cointegrated/rubert-tiny2 | lr недели 3: 0.0001
Эпох: 3 | batch: 16 | max_len: 256


In [3]:
# Поиск файла корпуса: рядом с ноутбуком, в папке проекта или загрузка в Colab
CANDIDATES = [Path("data/raw/corpus.jsonl"), Path("../data/raw/corpus.jsonl"),
              Path("corpus.jsonl")]
corpus_path = next((p for p in CANDIDATES if p.exists()), None)
if corpus_path is None:
    try:
        from google.colab import files
        print("Загрузите файл corpus.jsonl (data/raw/corpus.jsonl из проекта)")
        files.upload()
        corpus_path = Path("corpus.jsonl")
    except ImportError:
        raise FileNotFoundError("Не найден corpus.jsonl. Положите его рядом с ноутбуком или в data/raw/")

with corpus_path.open(encoding="utf-8") as f:
    df = pd.DataFrame([json.loads(line) for line in f if line.strip()])

# Подготовка точно такая же, как на неделе 3, чтобы тестовая выборка совпала
df["label_name"] = df["law"].str.replace(r"\s*\((Общая|Особенная) часть\)", "", regex=True)
df["input_text"] = df["title"].fillna("") + ". " + df["text"]
df = df[df["text"].str.split().str.len() >= 10]
df = df.drop_duplicates("input_text").reset_index(drop=True)

labels = sorted(df["label_name"].unique())
label2id = {name: i for i, name in enumerate(labels)}
id2label = {i: name for name, i in label2id.items()}
df["label"] = df["label_name"].map(label2id)

trainval_df, test_df = train_test_split(
    df, test_size=TEST_SIZE, stratify=df["label"], random_state=SEED)   # то же, что на неделе 3
train_df, val_df = train_test_split(
    trainval_df, test_size=VAL_SIZE, stratify=trainval_df["label"], random_state=SEED)
train_df, val_df, test_df = [d.reset_index(drop=True) for d in (train_df, val_df, test_df)]
print(f"train {len(train_df)}, val {len(val_df)}, test {len(test_df)}; классов: {len(labels)}")

train 2709, val 301, test 753; классов: 7


In [4]:
def compute_metrics(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision (macro)": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Recall (macro)": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "F1 (macro)": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def encode(texts):
    return tokenizer(list(texts), truncation=True, max_length=MAX_LEN)


class EncodedDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, targets):
        self.enc = encodings
        self.targets = list(targets)

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, i):
        item = {k: self.enc[k][i] for k in self.enc.keys()}
        item["labels"] = self.targets[i]
        return item


def collate(batch):
    targets = torch.tensor([b["labels"] for b in batch])
    feats = [{k: v for k, v in b.items() if k != "labels"} for b in batch]
    out = tokenizer.pad(feats, return_tensors="pt")
    out["labels"] = targets
    return out


train_ds = EncodedDataset(encode(train_df["input_text"]), train_df["label"])
val_ds = EncodedDataset(encode(val_df["input_text"]), val_df["label"])
test_ds = EncodedDataset(encode(test_df["input_text"]), test_df["label"])
val_loader = torch.utils.data.DataLoader(val_ds, batch_size=32, shuffle=False, collate_fn=collate)
test_loader = torch.utils.data.DataLoader(test_ds, batch_size=32, shuffle=False, collate_fn=collate)
print("Токенизация готова")

Токенизация готова


## 2. Функция одного эксперимента

`run_experiment(имя, скорость обучения, доля обучающих данных)` дообучает модель с нуля (с предобученных весов), считает метрики на валидации и тесте и сохраняет результат в кэш. Подвыборка для 50% и 75% берётся стратифицированно, с тем же seed.

In [5]:
RUNS_PATH = Path("report/week4_runs.json")
Path("report").mkdir(exist_ok=True)
CACHE = json.loads(RUNS_PATH.read_text(encoding="utf-8")) if RUNS_PATH.exists() else {}


@torch.no_grad()
def predict(model, loader):
    model.eval()
    logits = []
    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items() if k != "labels"}
        logits.append(model(**batch).logits.float().cpu())
    probs = torch.softmax(torch.cat(logits), dim=-1)
    return probs.argmax(-1).numpy(), probs.max(-1).values.numpy()


def run_experiment(name, lr, frac):
    if name in CACHE:
        print(f"[{name}] взято из кэша")
        return CACHE[name]
    print(f"\n=== {name}: lr={lr:.2e}, обучающих данных {frac:.0%} ===")
    set_seed(SEED)
    if frac < 1:
        idx = train_test_split(np.arange(len(train_df)), train_size=frac,
                               stratify=train_df["label"], random_state=SEED)[0].tolist()
    else:
        idx = list(range(len(train_df)))
    g = torch.Generator()
    g.manual_seed(SEED)
    loader = torch.utils.data.DataLoader(
        torch.utils.data.Subset(train_ds, idx), batch_size=BATCH_SIZE,
        shuffle=True, collate_fn=collate, generator=g)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=len(labels), id2label=id2label, label2id=label2id).to(DEVICE)
    no_decay = ["bias", "LayerNorm.weight"]
    groups = [
        {"params": [p for n, p in model.named_parameters()
                    if not any(nd in n for nd in no_decay)], "weight_decay": WEIGHT_DECAY},
        {"params": [p for n, p in model.named_parameters()
                    if any(nd in n for nd in no_decay)], "weight_decay": 0.0}]
    optimizer = torch.optim.AdamW(groups, lr=lr)
    total = len(loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, int(WARMUP_RATIO * total), total)

    history = []
    for epoch in range(1, EPOCHS + 1):
        model.train()
        t0, running = time.time(), 0.0
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            loss = model(**batch).loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            running += loss.item()
        vp, _ = predict(model, val_loader)
        tp, _ = predict(model, test_loader)
        history.append({
            "epoch": epoch, "train_loss": running / len(loader),
            "val_f1": f1_score(val_df["label"], vp, average="macro", zero_division=0),
            "test_f1": f1_score(test_df["label"], tp, average="macro", zero_division=0)})
        print(f"  эпоха {epoch}/{EPOCHS}: loss {history[-1]['train_loss']:.4f}, "
              f"val F1 {history[-1]['val_f1']:.4f}, {time.time() - t0:.0f} с")

    vp, _ = predict(model, val_loader)
    tp, tc = predict(model, test_loader)
    result = {"name": name, "lr": lr, "frac": frac, "n_train": len(idx), "history": history,
              "val": compute_metrics(val_df["label"], vp),
              "test": compute_metrics(test_df["label"], tp),
              "test_preds": tp.tolist(), "test_conf": [round(float(c), 4) for c in tc]}
    CACHE[name] = result
    RUNS_PATH.write_text(json.dumps(CACHE, ensure_ascii=False), encoding="utf-8")

    del model, optimizer, scheduler
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return result

## 3. Эксперимент 1: скорость обучения

Меняется только скорость обучения, всё остальное одинаково (100% обучающих данных, 3 эпохи, seed 42). Лучшее значение выбирается по F1 на **валидационной** выборке.

In [ ]:
LRS = [BASE_LR / 3, BASE_LR, BASE_LR * 5 / 3]
lr_runs = [run_experiment(f"lr={lr:.2e}|100%", lr, 1.0) for lr in LRS]
best_lr_run = max(lr_runs, key=lambda r: r["val"]["F1 (macro)"])
best_lr = best_lr_run["lr"]
print(f"\nЛучшая скорость обучения по валидации: {best_lr:.2e}")


=== lr=3.33e-05|100%: lr=3.33e-05, обучающих данных 100% ===


Loading weights: 100%|██████████| 55/55 [00:00<00:00, 3520.70it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the ch

  эпоха 1/3: loss 1.4338, val F1 0.7328, 280 с
  эпоха 2/3: loss 0.5389, val F1 0.9678, 276 с
  эпоха 3/3: loss 0.3228, val F1 0.9786, 279 с

=== lr=1.00e-04|100%: lr=1.00e-04, обучающих данных 100% ===


Loading weights: 100%|██████████| 55/55 [00:00<00:00, 11610.97it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the c

  эпоха 1/3: loss 0.9143, val F1 0.9747, 265 с
  эпоха 2/3: loss 0.1073, val F1 0.9944, 260 с
  эпоха 3/3: loss 0.0417, val F1 0.9927, 236 с

=== lr=1.67e-04|100%: lr=1.67e-04, обучающих данных 100% ===


Loading weights: 100%|██████████| 55/55 [00:00<00:00, 34705.39it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the c

  эпоха 1/3: loss 0.7241, val F1 0.9837, 230 с


## 4. Эксперимент 2: размер обучающей выборки

С лучшей скоростью обучения модель обучается на 50%, 75% и 100% обучающих данных.

In [ ]:
size_runs = [run_experiment(f"lr={best_lr:.2e}|{int(f * 100)}%", best_lr, f) for f in (0.5, 0.75)]
size_runs.append(best_lr_run)
print("Готово")

## 5. Сводная таблица экспериментов

In [ ]:
def to_row(group, r):
    return {"Эксперимент": group, "lr": f"{r['lr']:.2e}", "Доля данных": f"{r['frac']:.0%}",
            "Обучающих статей": r["n_train"],
            "Val F1": round(r["val"]["F1 (macro)"], 4),
            "Accuracy": round(r["test"]["Accuracy"], 4),
            "Precision": round(r["test"]["Precision (macro)"], 4),
            "Recall": round(r["test"]["Recall (macro)"], 4),
            "F1": round(r["test"]["F1 (macro)"], 4)}


rows = [to_row("скорость обучения", r) for r in lr_runs]
rows += [to_row("размер выборки", r) for r in size_runs]
summary = pd.DataFrame(rows)
summary.to_csv("report/week4_experiments.csv", index=False, encoding="utf-8-sig")
summary

## 6. Графики

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))

# 1. скорость обучения -> метрики на тесте
xs = [r["lr"] for r in lr_runs]
for key, style in (("Accuracy", "o-"), ("F1 (macro)", "s--")):
    axes[0].plot(xs, [r["test"][key] for r in lr_runs], style, label=key)
axes[0].set_xscale("log")
axes[0].set_xticks(xs)
axes[0].set_xticklabels([f"{x:.1e}" for x in xs])
axes[0].minorticks_off()
axes[0].set_xlabel("скорость обучения")
axes[0].set_ylabel("метрика на тесте")
axes[0].set_title("Влияние скорости обучения")
axes[0].legend()
axes[0].grid(alpha=0.3)

# 2. динамика по эпохам
for r in lr_runs:
    ep = [h["epoch"] for h in r["history"]]
    axes[1].plot(ep, [h["val_f1"] for h in r["history"]], marker="o", label=f"lr {r['lr']:.1e}")
axes[1].set_xticks(range(1, EPOCHS + 1))
axes[1].set_xlabel("эпоха")
axes[1].set_ylabel("F1 (macro) на валидации")
axes[1].set_title("Динамика F1 по эпохам")
axes[1].legend()
axes[1].grid(alpha=0.3)

# 3. размер выборки
sz = [r["frac"] * 100 for r in size_runs]
for key, style in (("Accuracy", "o-"), ("F1 (macro)", "s--")):
    axes[2].plot(sz, [r["test"][key] for r in size_runs], style, label=key)
axes[2].set_xticks(sz)
axes[2].set_xlabel("доля обучающих данных, %")
axes[2].set_ylabel("метрика на тесте")
axes[2].set_title("Влияние размера выборки")
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("report/week4_experiments.png", dpi=150)
plt.show()

## 7. Анализ ошибок

Анализируется лучшая модель (лучшая скорость обучения, 100% данных) на тестовой выборке. Каждой ошибке автоматически назначается основная причина по правилам, которые перечислены в коде. Правила проверяются по порядку, первое сработавшее определяет причину, поэтому порядок важен.

In [ ]:
best = best_lr_run
test_df["pred"] = best["test_preds"]
test_df["conf"] = best["test_conf"]

# признаки для объяснения ошибок (считаем для всех тестовых статей)
test_df["n_tokens"] = [len(ids) for ids in tokenizer(list(test_df["input_text"]), truncation=False)["input_ids"]]
test_df["n_words"] = test_df["text"].str.split().str.len()
test_df["noise"] = [1 - sum(ch.isalpha() or ch.isspace() for ch in t) / max(len(t), 1) for t in test_df["text"]]

errs = test_df[test_df["pred"] != test_df["label"]].copy()
print(f"Лучшая модель: lr {best['lr']:.2e}, accuracy {best['test']['Accuracy']:.4f}, "
      f"ошибок {len(errs)} из {len(test_df)}")

train_share = train_df["label_name"].value_counts(normalize=True)
errs["true_name"] = errs["label"].map(id2label)
errs["pred_name"] = errs["pred"].map(id2label)
pair_counts = Counter(zip(errs["true_name"], errs["pred_name"]))

SHORT_WORDS = 30    # короче этого порога текст считаем малоинформативным
NOISE_SHARE = 0.25  # доля цифр и символов в тексте, выше которой считаем текст «шумным»
RARE_SHARE = 0.10   # класс с долей в обучающей выборке ниже этой считаем редким
GENERIC = re.compile(r"основные понят|термины|общие положен|задачи|принципы|сфера|"
                     r"область примен|определени|международн|законодательство", re.I)

CAUSES = ["шум в тексте (числа, таблицы, артефакты разметки)",
          "короткий малоинформативный текст",
          "обрезка длинного текста (часть статьи не попала в модель)",
          "неоднозначность контекста (общая норма, встречается во многих кодексах)",
          "дисбаланс классов (редкий кодекс)",
          "пересечение лексики смежных отраслей права",
          "прочее"]


def primary_cause(r):
    if r["noise"] > NOISE_SHARE:
        return CAUSES[0]
    if r["n_words"] < SHORT_WORDS:
        return CAUSES[1]
    if r["n_tokens"] > MAX_LEN:
        return CAUSES[2]
    if GENERIC.search(str(r["title"])):
        return CAUSES[3]
    if train_share.get(r["true_name"], 0) < RARE_SHARE:
        return CAUSES[4]
    if pair_counts[(r["true_name"], r["pred_name"])] >= 2:
        return CAUSES[5]
    return CAUSES[6]


errs["cause"] = errs.apply(primary_cause, axis=1)
cause_table = errs["cause"].value_counts().rename("ошибок").to_frame()
cause_table["доля, %"] = (100 * cause_table["ошибок"] / len(errs)).round(1)
cause_table

In [ ]:
# Какие кодексы путаются чаще всего
pairs = pd.DataFrame(
    [(t, p, c) for (t, p), c in pair_counts.most_common(8)],
    columns=["на самом деле", "предсказано", "ошибок"])
pairs

In [ ]:
# Таблица не менее чем из 10 ошибок. Если у лучшей модели ошибок меньше 10,
# недостающие берутся из других экспериментов (колонка «модель»).
def snippet(text, n=170):
    t = " ".join(str(text).split())
    return t if len(t) <= n else t[:n] + "…"


examples, seen = [], set()
for i, r in errs.sort_values("conf", ascending=False).iterrows():
    examples.append({"модель": f"lr {best['lr']:.1e}, 100%", "idx": i, "pred": r["pred_name"], "conf": r["conf"]})
    seen.add(i)
if len(examples) < 10:
    for run in lr_runs + size_runs:
        for i, (p, c) in enumerate(zip(run["test_preds"], run["test_conf"])):
            if len(examples) >= 10:
                break
            if p != test_df.loc[i, "label"] and i not in seen:
                examples.append({"модель": f"lr {run['lr']:.1e}, {run['frac']:.0%}",
                                 "idx": i, "pred": id2label[p], "conf": c})
                seen.add(i)

rows = []
for k, e in enumerate(examples[:15], 1):
    r = test_df.loc[e["idx"]]
    cause = errs.loc[e["idx"], "cause"] if e["idx"] in errs.index else "ошибка только у другой модели"
    rows.append({"№": k, "Кодекс (верно)": id2label[r["label"]], "Предсказано": e["pred"],
                 "Уверенность": e["conf"], "Статья": f"ст. {r['article_number']}. {r['title']}",
                 "Фрагмент текста": snippet(r["text"]), "Причина": cause, "Модель": e["модель"]})
top_errors = pd.DataFrame(rows)
top_errors.to_csv("report/week4_top_errors.csv", index=False, encoding="utf-8-sig")
errs[["true_name", "pred_name", "conf", "title", "n_words", "n_tokens", "cause"]].to_csv(
    "report/week4_all_errors.csv", encoding="utf-8-sig")
pd.set_option("display.max_colwidth", 120)
top_errors

## 8. Причины ошибок и предложения по исправлению

Для каждой причины предложено решение. Часть из них можно проверить экспериментом на неделе 5-6, если останется время.

In [ ]:
FIXES = {
    CAUSES[0]: "Очищать текст до подачи в модель: убирать таблицы, числовые ссылки и служебные символы.",
    CAUSES[1]: "Добавлять к статье название главы или соседние статьи как контекст; совсем короткие статьи помечать как «неопределённые».",
    CAUSES[2]: "Брать начало и конец статьи вместо одного начала, либо резать на фрагменты и усреднять предсказания; на видеокарте увеличить max_len до 512.",
    CAUSES[3]: "Такие нормы («основные понятия», «принципы») почти одинаковы во всех кодексах, это естественная неоднозначность. Можно ввести класс «общая норма» или учитывать контекст соседних статей.",
    CAUSES[4]: "Взвешенная функция потерь (веса классов), увеличение числа примеров редких кодексов, стратифицированная выборка.",
    CAUSES[5]: "Дообучать дольше и на большем числе статей, добавить признаки контекста (раздел, глава), разобрать матрицу ошибок по этой паре кодексов.",
    CAUSES[6]: "Разобрать вручную: возможно, это новая причина, которую стоит добавить в правила.",
}
fix_table = cause_table.copy()
fix_table["что предлагается"] = [FIXES[c] for c in fix_table.index]
pd.set_option("display.max_colwidth", 200)
fix_table

## 9. Черновик аналитического отчёта

Ячейка ниже подставляет реальные цифры в текст и сохраняет его в `report/week4_analysis.md`. Это заготовка: перепишите выводы своими словами.

In [ ]:
def trend(vals):
    d = vals[-1] - vals[0]
    return "растёт" if d > 0.005 else ("падает" if d < -0.005 else "почти не меняется")


lr_f1 = [r["test"]["F1 (macro)"] for r in lr_runs]
sz_f1 = [r["test"]["F1 (macro)"] for r in size_runs]
sz_acc = [r["test"]["Accuracy"] for r in size_runs]
lr_names = ", ".join(f"{r['lr']:.1e}: F1 {r['test']['F1 (macro)']:.3f}" for r in lr_runs)
sz_names = ", ".join(f"{r['frac']:.0%}: accuracy {r['test']['Accuracy']:.3f}" for r in size_runs)
top_cause = cause_table.index[0] if len(cause_table) else "-"
top_pair = pairs.iloc[0] if len(pairs) else None

lines = [
    "# Анализ экспериментов и ошибок (неделя 4)", "",
    f"Модель: {MODEL_NAME}. Обучающих статей {len(train_df)}, валидационных {len(val_df)}, тестовых {len(test_df)}, "
    f"классов {len(labels)}.", "",
    "## 1. Скорость обучения",
    f"Результаты на тесте: {lr_names}.",
    f"Разброс F1 между лучшей и худшей скоростью: {100 * (max(lr_f1) - min(lr_f1)):.1f} п.п. "
    f"Лучшей по валидации оказалась скорость {best_lr:.1e}.", "",
    "## 2. Размер обучающей выборки",
    f"Результаты на тесте: {sz_names}.",
    f"При росте выборки с 50% до 100% качество {trend(sz_f1)} "
    f"(F1 {sz_f1[0]:.3f} -> {sz_f1[-1]:.3f}, изменение {100 * (sz_f1[-1] - sz_f1[0]):+.1f} п.п.).", "",
    "## 3. Анализ ошибок",
    f"У лучшей модели {len(errs)} ошибок на {len(test_df)} тестовых статьях "
    f"(accuracy {best['test']['Accuracy']:.3f}).",
    f"Самая частая причина: {top_cause} ({int(cause_table['ошибок'].iloc[0]) if len(cause_table) else 0} ошибок)."]
if top_pair is not None:
    lines.append(f"Чаще всего путаются «{top_pair['на самом деле']}» и «{top_pair['предсказано']}» "
                 f"({int(top_pair['ошибок'])} раз).")
lines += ["", "Распределение причин:"]
lines += [f"- {c}: {int(n)}" for c, n in cause_table["ошибок"].items()]
lines += ["", "## 4. Предложения по исправлению"]
lines += [f"- {c}: {FIXES[c]}" for c in cause_table.index]
text = "\n".join(lines)
Path("report/week4_analysis.md").write_text(text, encoding="utf-8")
print(text)